# Exercises XP Ninja: Advanced CNN Projects

This guided notebook follows the exercises on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts to support intuition or transfer to other AI topics.


## What you will learn
- Advanced CNN architectures and techniques
- Handling imbalanced datasets
- Transfer learning and fine tuning
- Model interpretability and visualization
- Building a robust classification pipeline


## What you will create
Five CNN based projects. Each addresses a specific challenge.


## Common setup

**As stated in the exercises**  
You will build and evaluate CNN models under different constraints.

**PREFILLED**  
Imports, versions, seed control, and a simple plot helper. Execute this first.


In [ ]:
# PREFILLED: just execute
import os, time, math, json, random, itertools
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

def plot_history(history, title="Training curves"):
    plt.figure(figsize=(6,4))
    if "accuracy" in history.history: plt.plot(history.history["accuracy"], label="acc")
    if "val_accuracy" in history.history: plt.plot(history.history["val_accuracy"], label="val_acc")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(6,4))
    if "loss" in history.history: plt.plot(history.history["loss"], label="loss")
    if "val_loss" in history.history: plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.tight_layout(); plt.show()

# Exercise 1: Multi label image classification

**As stated in the exercises**  
Choose a dataset where each image can have multiple labels. Build a CNN with a sigmoid output and use binary cross entropy. Handle dependencies between labels by experimenting with approaches such as recurrent layers or label embeddings.


### 1.1 Dataset interface

**PREFILLED**  
Expected CSV format: `filepath,labels` where `labels` is a `|` separated list. The code below parses the CSV and builds a multi hot vector per image based on the discovered vocabulary. If no CSV is provided, a toy multi label dataset is synthesized from CIFAR 10 using attributes such as `animal` and `vehicle` for demonstration.


In [ ]:
# PREFILLED: just execute
import pandas as pd

def build_multilabel_from_csv(csv_path, root_dir):
    df = pd.read_csv(csv_path)
    df['filepath'] = df['filepath'].astype(str)
    df['labels'] = df['labels'].astype(str)
    vocab = sorted(set(itertools.chain.from_iterable([s.split('|') for s in df['labels']])))
    tok = {t:i for i,t in enumerate(vocab)}
    y = np.zeros((len(df), len(vocab)), dtype=np.float32)
    for i, s in enumerate(df['labels']):
        for t in s.split('|'):
            y[i, tok[t]] = 1.0
    x_paths = [str(Path(root_dir)/p) for p in df['filepath']]
    return x_paths, y, vocab

def load_image(path, size=(160,160)):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, size)
    return img

def make_dataset_multilabel(x_paths, y, batch=32, shuffle=True, aug=False, size=(160,160)):
    x = tf.constant(x_paths)
    y = tf.constant(y, dtype=tf.float32)
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    def _map(p, lab):
        im = load_image(p, size)
        if aug:
            im = tf.image.random_flip_left_right(im)
            im = tf.image.random_brightness(im, 0.1)
            im = tf.image.random_contrast(im, 0.9, 1.1)
        return im, lab
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, len(x_paths)))
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

# Try to locate a CSV next to this notebook
CSV_CANDIDATES = [p for p in Path('.').glob('*.csv')]
if CSV_CANDIDATES:
    print("Found CSV:", CSV_CANDIDATES[0])
else:
    print("No CSV found. A toy dataset will be used if you run the fallback cell below.")

In [ ]:
# PREFILLED: just execute: toy multi label fallback using CIFAR 10
from tensorflow.keras.datasets import cifar10
(x_tr_c10, y_tr_c10), (x_te_c10, y_te_c10) = cifar10.load_data()
y_tr_c10 = y_tr_c10.ravel(); y_te_c10 = y_te_c10.ravel()
# attributes: animal vs vehicle, pet vs wild
attr = {
    0: ("vehicle","air"),     # airplane
    1: ("vehicle","land"),    # automobile
    2: ("animal","wild"),     # bird
    3: ("animal","pet"),      # cat
    4: ("animal","wild"),     # deer
    5: ("animal","pet"),      # dog
    6: ("animal","wild"),     # frog
    7: ("animal","pet"),      # horse as non pet but domesticated
    8: ("vehicle","water"),   # ship
    9: ("vehicle","land"),    # truck
}
vocab = sorted(list(set(itertools.chain.from_iterable(attr.values()))))
tok = {t:i for i,t in enumerate(vocab)}

def to_multi_hot(y):
    out = np.zeros((len(y), len(vocab)), dtype=np.float32)
    for i,c in enumerate(y):
        a,b = attr[int(c)]
        out[i, tok[a]] = 1.0
        out[i, tok[b]] = 1.0
    return out

y_tr_ml = to_multi_hot(y_tr_c10)
y_te_ml = to_multi_hot(y_te_c10)

def ds_from_numpy(x, y, size=(160,160), batch=64, aug=False):
    x = tf.convert_to_tensor(x)
    y = tf.convert_to_tensor(y, dtype=tf.float32)
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    def _map(im, lab):
        im = tf.image.convert_image_dtype(im, tf.float32)
        im = tf.image.resize(im, size)
        if aug:
            im = tf.image.random_flip_left_right(im)
            im = tf.image.random_brightness(im, 0.1)
        return im, lab
    return ds.shuffle(5000).map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds_ml = ds_from_numpy(x_tr_c10, y_tr_ml, aug=True)
val_ds_ml   = ds_from_numpy(x_te_c10, y_te_ml, aug=False)
print("Toy multi label dataset ready. Labels:", vocab)

### 1.2 Build a multi label CNN

**To-Do:** Create a CNN that outputs `len(vocab)` logits with sigmoid activation. Compile with `binary_crossentropy`. Train on your dataset and report micro and macro F1 on the validation set. Save the list of labels you used.


In [ ]:
# To-Do: define and train the multi label model
# inputs = layers.Input(shape=(160,160,3))
# ....
# outputs = layers.Dense(len(vocab), activation="sigmoid")(x)
# model_ml = models.Model(inputs, outputs)
# model_ml.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
# h_ml = model_ml.fit(train_ds_ml, validation_data=val_ds_ml, epochs=10)
# plot_history(h_ml, title="Exercise 1 multi label")

In [ ]:
# To-Do: compute micro and macro F1
# from sklearn.metrics import f1_score
# y_true, y_pred = [], []
# for xb, yb in val_ds_ml:
#     ....
# print("F1 micro:", f1_score(y_true.ravel(), y_pred.ravel(), average="micro"))
# print("F1 macro:", f1_score(y_true, y_pred, average="macro"))

### 1.3 Capture label dependencies

**To-Do:** Add a simple label dependency module. Example options: a Dense layer on logits to allow interaction, a small GRU over label dimension, or a learned label embedding with self attention. Train and compare F1 to the base model.


**Learning point**  
Multi label outputs model independent Bernoulli decisions. Explicitly modeling co occurrence can improve calibration and reduce contradictory predictions.


# Exercise 2: Class imbalance handling

**As stated in the exercises**  
Select a dataset with strong class imbalance. Train a CNN and address imbalance with oversampling, undersampling, or weighted loss. Evaluate with precision, recall, F1, and AUC ROC and compare to a baseline.


### 2.1 Simulate imbalance

**PREFILLED**  
We create an imbalanced subset of CIFAR 10 by downsampling one class.


In [ ]:
# PREFILLED: just execute
from tensorflow.keras.datasets import cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.ravel(); y_test = y_test.ravel()

minor_class = 2  # bird
keep_minor = 300
keep_major = 5000

idx_minor = np.where(y_train==minor_class)[0]
idx_major = np.where(y_train!=minor_class)[0]
sel = np.concatenate([np.random.choice(idx_minor, keep_minor, replace=False),
                      np.random.choice(idx_major, keep_major, replace=False)])
np.random.shuffle(sel)
x_imb = x_train[sel]; y_imb = y_train[sel]
print("Imbalanced set:", x_imb.shape, "minority ratio:", (y_imb==minor_class).mean())

### 2.2 Baseline model and metrics

**To-Do:** Train a baseline CNN on the imbalanced set without any correction. Compute precision, recall, F1 per class and macro average. Plot a normalized confusion matrix.


In [ ]:
# To-Do: train baseline on x_imb, y_imb
# base = tf.keras.Sequential([
#     ...
# ])
# base.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# h_base = base.fit(x_imb, y_imb, epochs=10, batch_size=128, validation_split=0.2, verbose=2)
# plot_history(h_base, title="Exercise 2 baseline")

In [ ]:
# To-Do: evaluate baseline with precision, recall, F1, and confusion matrix
# from sklearn.metrics import classification_report, confusion_matrix
# y_pred = ...
# print(classification_report(y_test, y_pred, digits=3))
# cm = confusion_matrix(y_test, y_pred, labels=list(range(10)))
# plt.figure(figsize=(6,6))
# plt.imshow(cm / cm.sum(axis=1, keepdims=True))
# plt.title("Normalized confusion matrix")
# plt.xlabel("pred"); plt.ylabel("true")
# plt.colorbar(); plt.tight_layout(); plt.show()

### 2.3 Apply imbalance techniques

**To-Do:** Train three new runs using: class weights, oversampling, and focal loss. Keep the architecture fixed. Compare the per class recall and macro F1.


In [ ]:
# To-Do: class weights
# unique, counts = np.unique(y_imb, return_counts=True)
# freq = ...
# cw = ...
# base_cw = ...
# base_cw.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# h_cw = base_cw.fit(x_imb, y_imb, epochs=10, batch_size=128, validation_split=0.2, class_weight=cw, verbose=2)

In [ ]:
# To-Do: oversampling with tf.data
# def ds_from_numpy(x, y, batch=128, aug=True):
#     ...
#         return ...
#     return ds.shuffle(len(x)).map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

# ds_minor = ds_from_numpy(x_imb[y_imb==minor_class], y_imb[y_imb==minor_class])
# ds_major = ds_from_numpy(x_imb[y_imb!=minor_class], y_imb[y_imb!=minor_class])
# ds_bal = tf.data.Dataset.sample_from_datasets([ds_minor, ds_major], weights=[0.5, 0.5]).prefetch(tf.data.AUTOTUNE)
# base_os = tf.keras.models.clone_model(base)
# base_os.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# h_os = base_os.fit(ds_bal, epochs=10, steps_per_epoch=math.ceil(len(x_imb)/128), verbose=2)

In [ ]:
# To-Do: focal loss
# def focal_loss(gamma=2.0, alpha=0.25):
#     def _loss(y_true, y_pred):
#         ...
#         return ...
#     return _loss
# base_fl = tf.keras.models.clone_model(base)
# base_fl.compile(optimizer="adam", loss=focal_loss(), metrics=["accuracy"])
# h_fl = base_fl.fit(x_imb, y_imb, epochs=10, batch_size=128, validation_split=0.2, verbose=2)

**Learning point**  
Accuracy hides minority performance. Macro or weighted F1, per class recall, and ROC AUC reveal improvements from imbalance handling.


# Exercise 3: Transfer learning and fine tuning

**As stated in the exercises**  
Use a pre trained backbone such as ResNet, EfficientNet, or Inception. Compare fine tuning strategies to training from scratch. Study the effect of learning rate on fine tuning.


### 3.1 Frozen backbone

**PREFILLED**  
MobileNetV2 is used for speed. The base is frozen. The head is trained first.


In [ ]:
# PREFILLED: just execute
IMG_FE = (160,160)
base = tf.keras.applications.MobileNetV2(input_shape=(IMG_FE[0], IMG_FE[1], 3),
                                         include_top=False, weights="imagenet")
base.trainable = False
preproc = tf.keras.applications.mobilenet_v2.preprocess_input

def build_head(num_classes=10):
    inputs = layers.Input(shape=(32,32,3))
    x = tf.image.resize(inputs, IMG_FE)
    x = preproc(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs)

from tensorflow.keras.datasets import cifar10
(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
y_tr = y_tr.ravel(); y_te = y_te.ravel()

model_frozen = build_head(10)
model_frozen.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_frozen = model_frozen.fit(x_tr, y_tr, epochs=5, batch_size=128, validation_split=0.2, verbose=2)
plot_history(hist_frozen, title="Frozen backbone head training")

### 3.2 Fine tune

**To-Do:** Unfreeze the top N layers of the backbone. Use a lower learning rate. Train for a few more epochs. Compare accuracy to the frozen model and to a scratch model.


In [ ]:
# To-Do: unfreeze and fine tune
# base.trainable = True
# for l in base.layers[:-40]:
#     l.trainable = False
# model_ft = build_head(10)  # rebuild to attach the now-trainable base
# model_ft.layers[3] = base  # reuse the same base instance
# model_ft.compile(...)
# hist_ft = model_ft.fit(...)
# plot_history(hist_ft, title="Fine tuned backbone")

**Learning point**  
Freeze then fine tune is stable and efficient. Use a lower learning rate for the backbone than for the head to avoid destroying pre trained features.


# Exercise 4: Model interpretability

**As stated in the exercises**  
Train a CNN and understand which regions drive predictions. Use Grad CAM and optionally LIME. Visualize attention maps and reason about relevance and bias.


### 4.1 Grad CAM helper

**PREFILLED**  
The function below computes Grad CAM for a given image and class index for models with a final conv layer.


In [ ]:
# PREFILLED: just execute
def grad_cam(model, img, class_index, layer_name=None):
    if layer_name is None:
        # find last conv layer
        for l in reversed(model.layers):
            if isinstance(l, layers.Conv2D):
                layer_name = l.name
                break
    conv_layer = model.get_layer(layer_name)
    grad_model = tf.keras.models.Model([model.inputs], [conv_layer.output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(tf.expand_dims(img, 0))
        loss = preds[:, class_index]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    cam = tf.reduce_sum(tf.multiply(weights, conv_out[0]), axis=-1)
    cam = tf.maximum(cam, 0) / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], img.shape[:2])
    return cam.numpy().squeeze()

def overlay_cam(img, cam, alpha=0.4):
    plt.figure(figsize=(4,4))
    plt.imshow(img.astype("uint8"))
    plt.imshow(cam, cmap="jet", alpha=alpha)
    plt.axis("off"); plt.tight_layout(); plt.show()

### 4.2 Run Grad CAM

**To-Do:** Choose a trained model and a few sample images. Generate Grad CAM maps for the predicted class. Inspect whether highlighted regions correspond to the object. Document your findings.


In [ ]:
# To-Do: pick a model and sample images, then visualize Grad CAM
# img = x_te[0]  # example
# pred = ...
# cam = ...
# overlay_cam(img, cam)

**Learning point**  
Grad CAM uses gradients of the target class flowing into the final conv layer to produce a heatmap. It helps verify that the network attends to object regions rather than background artifacts.


![image.png](attachment:image.png)

# Exercise 5: Building a robust classification pipeline

**As stated in the exercises**  
Create a complete pipeline for image classification including data loading, augmentation, training, evaluation, and saving. Optionally build a simple interface for predictions. Make the pipeline modular and robust.


### 5.1 Data module

**PREFILLED**  
A reusable loader using `image_dataset_from_directory`. It supports train and validation splits.


In [ ]:
# PREFILLED: just execute
def load_folder_dataset(root, img_size=(160,160), batch=32, val_split=0.2, seed=42):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        root, validation_split=val_split, subset="training", seed=seed,
        image_size=img_size, batch_size=batch, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        root, validation_split=val_split, subset="validation", seed=seed,
        image_size=img_size, batch_size=batch, label_mode="int"
    )
    class_names = train_ds.class_names
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)
    return train_ds, val_ds, class_names

### 5.2 Model factory

**To-Do:** Implement a model factory that can return a baseline CNN or a MobileNetV2 transfer model based on a string name. Use sensible defaults and allow passing the number of classes.


In [ ]:
# To-Do: implement model factory
# def make_model(kind, num_classes, img_size=(160,160)):
#     if kind == "baseline":
#         m = models.Sequential([
#             ...
#         ])
#         m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
#         return m
#     elif kind == "mobilenetv2":
#         base = tf.keras.applications.MobileNetV2(input_shape=(img_size[0], img_size[1], 3),
#                                                  include_top=False, weights="imagenet")
#         base.trainable = False
#         preproc = tf.keras.applications.mobilenet_v2.preprocess_input
#         inputs = layers.Input(shape=(img_size[0], img_size[1], 3))
#         x = preproc(inputs)
#         x = base(x, training=False)
#         x = layers.GlobalAveragePooling2D()(x)
#         x = layers.Dense(128, activation="relu")(x)
#         outputs = layers.Dense(num_classes, activation="softmax")(x)
#         m = models.Model(inputs, outputs)
#         m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
#         return m
#     else:
#         raise ValueError("Unknown kind")

### 5.3 Training, evaluation, saving

**To-Do:** Train a selected model on a folder dataset. Evaluate on validation data. Save the model and class names to disk and reload them for a test prediction. Add simple error handling for unknown inputs.


In [ ]:
# Pre-filled: if you want to do an end to end run + saving
# root = "/path/to/your/dataset"
# train_ds, val_ds, class_names = load_folder_dataset(root)
# model = make_model("mobilenetv2", num_classes=len(class_names))
# cb = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
# hist = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cb)
# plot_history(hist, title="Exercise 5 pipeline")
# model.save("./data/robust_cnn_savedmodel")
# json.dump(class_names, open("./data/robust_cnn_classes.json", "w"))
# # Reload and predict a single image
# m2 = tf.keras.models.load_model("./data/robust_cnn_savedmodel")
# classes = json.load(open("./data/robust_cnn_classes.json"))
# # path_to_img = "/path/to/test.jpg"
# # im = load_image(path_to_img, size=(160,160)).numpy()
# # pr = m2.predict(im[None], verbose=0)[0]
# # print(classes[int(np.argmax(pr))], float(np.max(pr)))

**Learning point**  
A robust pipeline is modular and testable. Keep data loading, modeling, and evaluation separate. Save models with their label maps. Validate inputs and fail safely.
